# 1. Executive Summary

This project investigates how computational text analysis can reveal differences in writing and storytelling styles in science fiction.

A dataset of 20 texts from Project Gutenberg was prepared and explored using Python. Initial analysis examined text lengths and common word frequencies. Six works by three authors (Andre Norton, Leigh Brackett and Marion Zimmer Bradley) were selected for closer comparison.

The investigation focused on three features: sentence length, estimated dialogue proportions and the use of common function words.

The results showed differences both between authors and between individual works. Norton’s selected works had the longest sentences on average, while Brackett’s had the shortest. Estimated dialogue proportions varied considerably, even between works by the same author.

Function-word usage also suggested a stylistic signature: works by the same author tended to be more similar, with a mean pairwise distance (Euclidean) between function-word frequency vectors of 11.63, compared with 21.42 for works by different authors.

Overall, computational analysis revealed some useful stylistic patterns. However, the findings are exploratory, given the small sample of works and the limitations of automated text analysis.

## 2. Introduction

Science fiction is well suited to the study of writing style, since authors can take very different approaches to language, dialogue and storytelling when building imagined worlds.

This project investigates a dataset of 20 science-fiction texts from Project Gutenberg, with a particular focus on works by women authors. The collection includes novels, shorter stories and a multi-author anthology. This dataset was chosen because it provides a range of writing styles that can be explored using computational methods.

**The central research question is:**

> To what extent can computational text analysis reveal differences in the linguistic and narrative styles of selected science-fiction authors?

I expected to find measurable differences in sentence structure, dialogue usage and common word frequencies. I was also interested in whether works by the same author would share recognisable patterns despite differences in their stories.

Python was used to prepare the texts, explore the dataset and compare selected writing features. The aim is to investigate possible stylistic differences rather than make definitive claims about individual authors.

## 3. Data Cleaning

The dataset consists of 20 Project Gutenberg text files (`.txt`) selected to investigate writing and storytelling styles in science fiction, with a particular focus on women authors. It includes novels, shorter works and one multi-author anthology (*Astounding Stories of Super-Science, May 1930*). Data cleaning aims to remove Project Gutenberg headers, footers and other non-literary material so that the texts are ready for analysis.

### 3.1 Checking the Dataset

The first step counts the `.txt` files in the dataset to confirm that Python has found all 20 books, which helps detect missing or incorrectly stored files before data cleaning begins. However, locating a file does not guarantee that it can be opened and imported successfully.

In [1]:
from pathlib import Path

book_folder = Path("Gutenberg_books")
book_files = sorted(book_folder.glob("*.txt"))

print("Number of books found:", len(book_files))

Number of books found: 20


### 3.2 Verifying File Accessibility

In [2]:
for book in book_files:
    try:
        with open(book, encoding="utf-8-sig") as file:
            content = file.read()

        print(book.name, "- Successfully opened")

    except (OSError, UnicodeError) as error:
        print(book.name, "- Error:", error)

1250-0.txt - Successfully opened
29019-0.txt - Successfully opened
84-0.txt - Successfully opened
pg16921.txt - Successfully opened
pg18247.txt - Successfully opened
pg18458.txt - Successfully opened
pg18846.txt - Successfully opened
pg19029.txt - Successfully opened
pg19090.txt - Successfully opened
pg19145.txt - Successfully opened
pg19651.txt - Successfully opened
pg19726.txt - Successfully opened
pg20788.txt - Successfully opened
pg20796.txt - Successfully opened
pg22544.txt - Successfully opened
pg23301.txt - Successfully opened
pg25550.txt - Successfully opened
pg29809.txt - Successfully opened
pg30427.txt - Successfully opened
pg32664.txt - Successfully opened


<table style="width: 100%; border-collapse: separate; border-spacing: 24px 0; margin-left: -24px;">
  <tr>
    <td style="width: 50%; vertical-align: top; padding: 0;">
      <h4 style="text-align: left;">Observation</h4>
      <p>
        Python identified 20 <code>.txt</code> files, matching the expected number of works.
        All 20 files could also be opened using the <code>utf-8-sig</code> encoding.
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; padding: 0;">
      <h4 style="text-align: left;">Implication</h4>
      <p>
        The files are accessible for further processing, although this does not confirm
        that their contents are consistently formatted or ready for analysis.
      </p>
    </td>
  </tr>
</table>

### 3.3 Inspecting a Sample File

A sample file, `84-0.txt` (*Frankenstein* by Mary Shelley), is inspected in more detail before the text extraction procedure is developed.

In [3]:
book_path = Path("Gutenberg_books") / "84-0.txt"

with open(book_path, encoding="utf-8-sig") as file:
    text = file.read()

print("File loaded successfully!")

File loaded successfully!


To check for non-literary content, the first and last 1,000 characters of the imported text are displayed. This shows that the file contains additional material, including publication dates, language information, character encoding and other metadata. Because this investigation compares literary writing styles, including sentence length and structure, this material must be identified and removed before analysis to avoid skewing the results.

In [4]:
print(text[:1000])

The Project Gutenberg eBook of Frankenstein, by Mary Wollstonecraft (Godwin) Shelley

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at
www.gutenberg.org. If you are not located in the United States, you
will have to check the laws of the country where you are located before
using this eBook.

Title: Frankenstein
       or, The Modern Prometheus

Author: Mary Wollstonecraft (Godwin) Shelley

Release Date: 31, 1993 [eBook #84]
[Most recently updated: November 13, 2020]

Language: English

Character set encoding: UTF-8

Produced by: Judith Boss, Christy Phillips, Lynn Hanninen, and David Meltzer. HTML version by Al Haines.
Further corrections by Menno de Leeuw.

*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN ***




Frankenstein;

or, the Modern Prom

In [5]:
print(text[-1000:])

w.gutenberg.org/donate

Section 5. General Information About Project Gutenberg-tm electronic works

Professor Michael S. Hart was the originator of the Project
Gutenberg-tm concept of a library of electronic works that could be
freely shared with anyone. For forty years, he produced and
distributed Project Gutenberg-tm eBooks with only a loose network of
volunteer support.

Project Gutenberg-tm eBooks are often created from several printed
editions, all of which are confirmed as not protected by copyright in
the U.S. unless a copyright notice is included. Thus, we do not
necessarily keep eBooks in compliance with any particular paper
edition.

Most people start at our website which has the main PG search
facility: www.gutenberg.org

This website includes information about Project Gutenberg-tm,
including how to make donations to the Project Gutenberg Literary
Archive Foundation, how to help produce our new eBooks, and how to
subscribe to our email newsletter to hear about new eBooks.




<table style="width: 100%; border-collapse: separate; border-spacing: 24px 0; margin-left: -24px;">
  <tr>
    <td style="width: 50%; vertical-align: top; padding: 0;">
      <h4 style="text-align: left;">Observation</h4>
      <p style="text-align: left;">
        The start and end of <em>Frankenstein</em> show that the raw text includes Project Gutenberg
        publication details and licensing information as well as the novel itself, none of which is
        part of Shelley’s writing.
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; padding: 0;">
      <h4 style="text-align: left;">Implication</h4>
      <p style="text-align: left;">
        Leaving this material in could affect vocabulary counts, sentence-length calculations and
        other style comparisons, so the text needs to be cleaned before further analysis.
      </p>
    </td>
  </tr>
</table>

### 3.4 Locating the Gutenberg Markers

The next step is to identify the boundaries of the eBook content, starting with *Frankenstein*. The Gutenberg header, footer and licensing information are not part of the author's writing, and including them could skew comparisons in the analysis. The start and end markers will be located using Python's `.find()` method (Python Software Foundation, n.d.-b). Although these markers identify the boundaries of the eBook content, additional material may still be present within them.

In [6]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_position = text.find(start_marker)
end_position = text.find(end_marker)

print("Start marker position:", start_position)
print("End marker position:", end_position)

Start marker position: 904
End marker position: 420306


For the *Frankenstein* test case, the search found the Project Gutenberg start marker at character position 904 and the end marker at position 420,306. Both markers appeared in the expected order, which indicates clear boundaries between the novel and the surrounding Gutenberg text. These boundaries can be used to remove the header and footer, although the text between them still needs to be inspected for any remaining content that is not part of the literary work.

### 3.5 Marker Variations Across the Dataset

Manual inspection of the wider dataset showed that both markers vary. The start of the eBook content is marked by either “`*** START OF THE PROJECT GUTENBERG EBOOK`” or “`*** START OF THIS PROJECT GUTENBERG EBOOK`”, and the end by either “`*** END OF THE PROJECT GUTENBERG EBOOK`” or “`*** END OF THIS PROJECT GUTENBERG EBOOK`”. The initial search checks for both variations across all 20 files.

In [7]:
start_markers = [
    "*** START OF THIS PROJECT GUTENBERG EBOOK",
    "*** START OF THE PROJECT GUTENBERG EBOOK"
]

end_markers = [
    "*** END OF THIS PROJECT GUTENBERG EBOOK",
    "*** END OF THE PROJECT GUTENBERG EBOOK"
]

for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    start_positions = [content.find(marker) for marker in start_markers]
    end_positions = [content.find(marker) for marker in end_markers]

    print(book.name, "Start:", start_positions, "End:", end_positions)

1250-0.txt Start: [-1, 727] End: [-1, 108837]
29019-0.txt Start: [-1, 789] End: [-1, 17485]
84-0.txt Start: [-1, 904] End: [-1, 420306]
pg16921.txt Start: [447, -1] End: [337250, -1]
pg18247.txt Start: [-1, 686] End: [-1, 1000734]
pg18458.txt Start: [413, -1] End: [348761, -1]
pg18846.txt Start: [422, -1] End: [136135, -1]
pg19029.txt Start: [444, -1] End: [34151, -1]
pg19090.txt Start: [432, -1] End: [190230, -1]
pg19145.txt Start: [430, -1] End: [353370, -1]
pg19651.txt Start: [441, -1] End: [320060, -1]
pg19726.txt Start: [482, -1] End: [238516, -1]
pg20788.txt Start: [432, -1] End: [351576, -1]
pg20796.txt Start: [453, -1] End: [261522, -1]
pg22544.txt Start: [-1, -1] End: [-1, -1]
pg23301.txt Start: [429, -1] End: [36406, -1]
pg25550.txt Start: [443, -1] End: [321261, -1]
pg29809.txt Start: [512, -1] End: [457450, -1]
pg30427.txt Start: [468, -1] End: [122697, -1]
pg32664.txt Start: [439, -1] End: [132478, -1]


The initial search identified one of the two Gutenberg marker variations in 19 files, while one file matched neither. Each result contains two values, corresponding to the marker variations in the order they were searched. For example, `Start: [-1, 727] End: [-1, 108837]` indicates that the first variation returned -1 (no match), while the second was found at character positions 727 (start) and 108,837 (end). This identified the matching variation for each of the 19 files with recognised markers. The distribution is summarised below.


<table style="text-align:left;">
  <thead>
    <tr>
      <th>Marker variation</th>
      <th>Number of files</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>THIS PROJECT GUTENBERG EBOOK</td>
      <td>15</td>
    </tr>
    <tr>
      <td>THE PROJECT GUTENBERG EBOOK</td>
      <td>4</td>
    </tr>
    <tr>
      <td>Neither recognised</td>
      <td>1</td>
    </tr>
    <tr>
      <td><strong>Total</strong></td>
      <td><strong>20</strong></td>
    </tr>
  </tbody>
</table>


### 3.6 Investigating Inconsistent Gutenberg Markers

The initial search identified both start and end markers in 19 of the 20 files. However, `pg22544.txt` returned `[-1, -1]` for both searches, indicating that neither marker variation was found.

Manual inspection of this file, *A World Is Born*, revealed a slightly different start marker:

`***START OF THE PROJECT GUTENBERG EBOOK A WORLD IS BORN***`

Unlike the previously identified variations, there is no space between the three asterisks and the word `START`. Because Python's `.find()` method searches for an exact sequence of characters, this spacing difference explains why the original search failed.

To address this, a third start-marker variation has been added to the search criteria, along with a corresponding end-marker variation for testing. The amended code below repeats the search across all 20 files to check whether both boundaries can now be identified consistently.

In [8]:

# Include the additional Gutenberg marker variations
start_markers = [
    "*** START OF THIS PROJECT GUTENBERG EBOOK",
    "*** START OF THE PROJECT GUTENBERG EBOOK",
    "***START OF THE PROJECT GUTENBERG EBOOK"
]

end_markers = [
    "*** END OF THIS PROJECT GUTENBERG EBOOK",
    "*** END OF THE PROJECT GUTENBERG EBOOK",
    "***END OF THE PROJECT GUTENBERG EBOOK"
]

# Search all 20 files for the start and end markers
for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    start_positions = [content.find(marker) for marker in start_markers]
    end_positions = [content.find(marker) for marker in end_markers]

    print(book.name, "Start:", start_positions, "End:", end_positions)


1250-0.txt Start: [-1, 727, -1] End: [-1, 108837, -1]
29019-0.txt Start: [-1, 789, -1] End: [-1, 17485, -1]
84-0.txt Start: [-1, 904, -1] End: [-1, 420306, -1]
pg16921.txt Start: [447, -1, -1] End: [337250, -1, -1]
pg18247.txt Start: [-1, 686, -1] End: [-1, 1000734, -1]
pg18458.txt Start: [413, -1, -1] End: [348761, -1, -1]
pg18846.txt Start: [422, -1, -1] End: [136135, -1, -1]
pg19029.txt Start: [444, -1, -1] End: [34151, -1, -1]
pg19090.txt Start: [432, -1, -1] End: [190230, -1, -1]
pg19145.txt Start: [430, -1, -1] End: [353370, -1, -1]
pg19651.txt Start: [441, -1, -1] End: [320060, -1, -1]
pg19726.txt Start: [482, -1, -1] End: [238516, -1, -1]
pg20788.txt Start: [432, -1, -1] End: [351576, -1, -1]
pg20796.txt Start: [453, -1, -1] End: [261522, -1, -1]
pg22544.txt Start: [-1, -1, 505] End: [-1, -1, 47993]
pg23301.txt Start: [429, -1, -1] End: [36406, -1, -1]
pg25550.txt Start: [443, -1, -1] End: [321261, -1, -1]
pg29809.txt Start: [512, -1, -1] End: [457450, -1, -1]
pg30427.txt Start

### 3.7 Validation of the Revised Marker Search

The amended search identified both Gutenberg start and end markers in all 20 text files. In particular, `pg22544.txt`, which previously returned `[-1, -1]`, now returned `[-1, -1, 505]` for the start marker and `[-1, -1, 47993]` for the end marker. This confirms that the third variation, which omits the space after the opening asterisks, is used in both the header and footer of this file.

The final results show three start-marker variations across the dataset:

| Marker variation | Number of files |
|---|---:|
| `*** START OF THIS PROJECT GUTENBERG EBOOK` | 15 |
| `*** START OF THE PROJECT GUTENBERG EBOOK` | 4 |
| `***START OF THE PROJECT GUTENBERG EBOOK` | 1 |
| **Total** | **20** |

All 20 files now have identifiable start and end boundaries, so the cleaning process can proceed using a consistent set of search criteria. However, the text between these boundaries still needs to be inspected for any remaining material that is not part of the literary work.

### 3.8 Extracting the Gutenberg eBook Content

With the start and end markers identified in all 20 files, the next stage is to extract the text between these boundaries. This removes the Project Gutenberg header and footer, which are not part of the literary work and could affect later analysis.

The extraction will first be tested on *Frankenstein* before being applied to the complete dataset. The text is extracted from the line after the start marker up to the beginning of the end marker, so the marker text itself is excluded.

Further inspection will still be needed to identify any remaining non-literary material, such as tables of contents, publication information or introductory notes.

In [9]:
# Load Frankenstein as the sample text
book_path = book_folder / "84-0.txt"

with open(book_path, encoding="utf-8-sig") as file:
    content = file.read()

# Find the matching start and end markers
start_position = next(
    content.find(marker)
    for marker in start_markers
    if marker in content
)

end_position = next(
    content.find(marker)
    for marker in end_markers
    if marker in content
)

# Move past the complete start-marker line
text_start = content.find("\n", start_position) + 1

# Extract the text and remove leading/trailing whitespace
extracted_text = content[text_start:end_position].strip()

print("Original characters:", len(content))
print("Extracted characters:", len(extracted_text))


Original characters: 438876
Extracted characters: 419335


In [10]:

# Display the first 1000 characters
print("BEGINNING OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT:")
print("-" * 60)
print(extracted_text[:1000].lstrip())
print("-" * 60)
print("END OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT")

# Add a gap between sections
print("\n" * 2)

# Display the last 1000 characters
print("BEGINNING OF LAST 1000 CHARACTERS OF EXTRACTED TEXT:")
print("-" * 60)
print(extracted_text[-1000:].rstrip())
print("-" * 60)
print("END OF LAST 1000 CHARACTERS OF EXTRACTED TEXT")


BEGINNING OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT:
------------------------------------------------------------
Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chapter 11
 Chapter 12
 Chapter 13
 Chapter 14
 Chapter 15
 Chapter 16
 Chapter 17
 Chapter 18
 Chapter 19
 Chapter 20
 Chapter 21
 Chapter 22
 Chapter 23
 Chapter 24




Letter 1

_To Mrs. Saville, England._


St. Petersburgh, Dec. 11th, 17—.


You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
forebodings. I arrived here yesterday, and my first task is to assure
my dear sister of my welfare and increasing confidence in the success
of my undertaking.

I am already far north of London, and as I walk in the streets of
Petersburgh, I feel a cold northern

### 3.9 Remaining Non-Literary Content and Cleaning Limitations

The first and last 1,000 characters of the extracted *Frankenstein* text suggest that the Project Gutenberg header and footer have been removed. However, the start of the text still contains the title, author information and a table of contents listing the letters and chapters before the narrative begins at "Letter 1".

Although the table of contents is part of the published eBook, it is not part of the novel itself, and retaining it adds non-narrative content that may affect word-frequency and sentence-length measurements.

Removing this material programmatically is difficult because the 20 books may contain different types and arrangements of introductory content. Writing individual cleaning rules for each book would add complexity and risk introducing inconsistencies or accidentally removing genuine content.

The cleaning process will therefore prioritise the consistent removal of identifiable Project Gutenberg headers and footers across all 20 files. Any remaining content is treated as a limitation of the dataset and considered when interpreting the results.

### 3.10 Applying Gutenberg Boundary Cleaning to the Complete Dataset

With the Gutenberg start and end markers identified in all 20 files, the extraction process can now be applied across the complete dataset.

The code below uses the identified marker variations to locate the beginning and end of each eBook, then extracts the text between these boundaries, excluding the marker lines and the surrounding administrative information. The extracted texts are stored in a Python dictionary, so each book can be accessed by its filename without modifying the original files.

A validation check then confirms that all 20 books have been extracted and that each resulting text contains content. Remaining front matter, such as tables of contents and introductory information, is treated as a limitation in the subsequent analysis.

In [11]:

# Store the extracted text for each book
extracted_books = {}

for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    # Find recognised Gutenberg marker positions
    start_positions = [
        content.find(marker) for marker in start_markers
    ]

    end_positions = [
        content.find(marker) for marker in end_markers
    ]

    # Exclude unsuccessful searches (-1)
    valid_starts = [pos for pos in start_positions if pos != -1]
    valid_ends = [pos for pos in end_positions if pos != -1]

    if valid_starts and valid_ends:
        start_position = min(valid_starts)
        end_position = min(valid_ends)

        # Locate the end of the start-marker line
        line_end = content.find("\n", start_position)

        # Check that the boundaries are in the correct order
        if line_end != -1 and line_end < end_position:
            extracted_text = content[line_end + 1:end_position].strip()

            extracted_books[book.name] = extracted_text
        else:
            print("Invalid boundaries:", book.name)
    else:
        print("Missing markers:", book.name)

print("Books successfully extracted:", len(extracted_books))
print("Total files:", len(book_files))


Books successfully extracted: 20
Total files: 20


### 3.11 Validating the Extracted Dataset

Following extraction, the dataset is checked to confirm that all 20 files have been processed.

The validation examines the number of extracted books, the length of each extracted text and whether any files returned empty results. This provides an initial check that the automated cleaning process has produced usable text for subsequent analysis.

Although these checks cannot guarantee that every text contains only narrative content, they can identify obvious extraction failures before the Exploratory Data Analysis stage.

In [12]:

# Display the character count for each extracted book
for filename, book_text in extracted_books.items():
    print(filename, "- Characters extracted:", len(book_text))

# Check for empty extracted texts
empty_books = [
    filename
    for filename, book_text in extracted_books.items()
    if not book_text
]

print("\nVALIDATION SUMMARY")
print("-" * 40)
print("Total source files:", len(book_files))
print("Books extracted:", len(extracted_books))
print("Empty extracted texts:", len(empty_books))

if len(extracted_books) == len(book_files) and not empty_books:
    print("Validation passed: all files contain extracted text.")
else:
    print("Validation requires further investigation.")
    print("Empty files:", empty_books)


1250-0.txt - Characters extracted: 108043
29019-0.txt - Characters extracted: 16622
84-0.txt - Characters extracted: 419335
pg16921.txt - Characters extracted: 336739
pg18247.txt - Characters extracted: 999981
pg18458.txt - Characters extracted: 348286
pg18846.txt - Characters extracted: 135647
pg19029.txt - Characters extracted: 33637
pg19090.txt - Characters extracted: 189734
pg19145.txt - Characters extracted: 352871
pg19651.txt - Characters extracted: 319551
pg19726.txt - Characters extracted: 237959
pg20788.txt - Characters extracted: 351073
pg20796.txt - Characters extracted: 260997
pg22544.txt - Characters extracted: 47423
pg23301.txt - Characters extracted: 35910
pg25550.txt - Characters extracted: 320747
pg29809.txt - Characters extracted: 456867
pg30427.txt - Characters extracted: 122157
pg32664.txt - Characters extracted: 131966

VALIDATION SUMMARY
----------------------------------------
Total source files: 20
Books extracted: 20
Empty extracted texts: 0
Validation passed: 

### 3.12 Associating Gutenberg Filenames with Book Metadata

During the initial data cleaning, the extracted texts were identified by their original Project Gutenberg filenames. Associating these filenames with book titles and authors will make the results easier to interpret and the visualisations more meaningful.

Project Gutenberg provides an online catalogue containing eBook titles, authors and unique catalogue numbers (Project Gutenberg, n.d.-b), and these numbers can be identified within the filenames of the downloaded texts.

An automated approach using the Gutendex API, which retrieves Project Gutenberg metadata by catalogue ID (Johnson, n.d.), was attempted first, but repeated connection timeouts prevented it from being completed.

As an alternative, the titles and author names were identified manually from the downloaded text files and cross-referenced with the catalogue records on the Project Gutenberg website. This information was stored in a Python dictionary, using the original filenames as unique keys (Python Software Foundation, n.d.).

A validation check confirmed that all 20 extracted texts had corresponding metadata entries. The original filenames were retained to maintain traceability between the source files and the titles and authors used in later analysis.

In [13]:

# Extract the Gutenberg catalogue number from each filename
book_ids = {}

for filename in extracted_books:
    gutenberg_id = int(
        filename.replace("pg", "").replace("-0.txt", "").replace(".txt", "")
    )
    book_ids[filename] = gutenberg_id

# Display the filename-to-ID mapping
for filename, gutenberg_id in book_ids.items():
    print(filename, "->", gutenberg_id)


1250-0.txt -> 1250
29019-0.txt -> 29019
84-0.txt -> 84
pg16921.txt -> 16921
pg18247.txt -> 18247
pg18458.txt -> 18458
pg18846.txt -> 18846
pg19029.txt -> 19029
pg19090.txt -> 19090
pg19145.txt -> 19145
pg19651.txt -> 19651
pg19726.txt -> 19726
pg20788.txt -> 20788
pg20796.txt -> 20796
pg22544.txt -> 22544
pg23301.txt -> 23301
pg25550.txt -> 25550
pg29809.txt -> 29809
pg30427.txt -> 30427
pg32664.txt -> 32664


In [14]:
# Book titles and authors manually checked against Project Gutenberg catalogue

book_metadata = {
    "1250-0.txt": {"Title": "Anthem", "Author": "Ayn Rand"},
    "29019-0.txt": {"Title": "All Cats Are Gray", "Author": "Andre Norton"},
    "84-0.txt": {"Title": "Frankenstein; Or, The Modern Prometheus", "Author": "Mary Shelley"},
    "pg16921.txt": {"Title": "Plague Ship", "Author": "Andre Norton"},
    "pg18247.txt": {"Title": "The Last Man", "Author": "Mary Shelley"},
    "pg18458.txt": {"Title": "Star Born", "Author": "Andre Norton"},
    "pg18846.txt": {"Title": "Voodoo Planet", "Author": "Andre Norton"},
    "pg19029.txt": {"Title": "The Gifts of Asti", "Author": "Andre Norton"},
    "pg19090.txt": {"Title": "Star Hunter", "Author": "Andre Norton"},
    "pg19145.txt": {"Title": "The Time Traders", "Author": "Andre Norton"},
    "pg19651.txt": {"Title": "Key Out of Time", "Author": "Andre Norton"},
    "pg19726.txt": {"Title": "The Door Through Space", "Author": "Marion Zimmer Bradley"},
    "pg20788.txt": {"Title": "Storm Over Warlock", "Author": "Andre Norton"},
    "pg20796.txt": {"Title": "The Colors of Space", "Author": "Marion Zimmer Bradley"},
    "pg22544.txt": {"Title": "A World Is Born", "Author": "Leigh Brackett"},
    "pg23301.txt": {"Title": "Each Man Kills", "Author": "Victoria Glad"},
    "pg25550.txt": {"Title": "The Defiant Agents", "Author": "Andre Norton"},
    "pg29809.txt": {"Title": "Astounding Stories of Super-Science, May 1930", "Author": "Various"},
    "pg30427.txt": {"Title": "The Lost Kafoozalum", "Author": "Pauline Ashwell"},
    "pg32664.txt": {"Title": "Black Amazon of Mars", "Author": "Leigh Brackett"}
}

# Check that every extracted text has a metadata entry
missing_metadata = set(extracted_books) - set(book_metadata)

print("Metadata entries:", len(book_metadata))
print("Missing metadata:", len(missing_metadata))

if missing_metadata:
    print("Files needing investigation:", sorted(missing_metadata))
else:
    print("Validation passed: all extracted texts have metadata.")


Metadata entries: 20
Missing metadata: 0
Validation passed: all extracted texts have metadata.


### 3.13 Final Data Cleaning Validation

The automated extraction processed all 20 Project Gutenberg text files. Validation confirmed that each file produced a non-empty extracted text, with no missing or empty results.

The cleaning process accounted for three Gutenberg marker formats, so text outside the recognised start and end boundaries was excluded from all 20 files. The original source files were preserved, and the extracted texts were stored in a Python dictionary for later analysis.

Some books may retain additional non-literary material, such as tables of contents and publication information, and this limitation is documented. The dataset is now suitable for initial exploratory analysis, provided these limitations are considered when interpreting stylistic measurements.

## 4. Exploratory Data Analysis

### 4.1 Dataset Overview

Following data cleaning, the 20 extracted eBooks are available for exploratory analysis. This section investigates the general characteristics of the dataset, identifies differences between the texts and highlights potential issues that may influence later analysis.

The extracted texts have been associated with their book titles and authors, so results can be presented with meaningful labels while the original Project Gutenberg filenames are retained for reference.

The initial exploration focuses on character counts and approximate word counts. These measurements indicate the length of each text and help establish whether comparisons between works will require normalisation to account for differences in size.

In [15]:

import pandas as pd

# Calculate text lengths and associate each file with its metadata
book_lengths = []

for filename, book_text in extracted_books.items():
    book_lengths.append({
        "Filename": filename,
        "Title": book_metadata[filename]["Title"],
        "Author": book_metadata[filename]["Author"],
        "Characters": len(book_text),
        "Words": len(book_text.split())
    })

# Create and display the dataset overview
books_df = pd.DataFrame(book_lengths)

display(books_df)

,Filename,Title,Author,Characters,Words
0,1250-0.txt,Anthem,Ayn Rand,108043,19153
1,29019-0.txt,All Cats Are Gray,Andre Norton,16622,2829
2,84-0.txt,"Frankenstein; Or, The Modern Prometheus",Mary Shelley,419335,75042
3,pg16921.txt,Plague Ship,Andre Norton,336739,60064
4,pg18247.txt,The Last Man,Mary Shelley,999981,174621
5,pg18458.txt,Star Born,Andre Norton,348286,62407
6,pg18846.txt,Voodoo Planet,Andre Norton,135647,23815
7,pg19029.txt,The Gifts of Asti,Andre Norton,33637,6119
8,pg19090.txt,Star Hunter,Andre Norton,189734,34014
9,pg19145.txt,The Time Traders,Andre Norton,352871,64368


#### Initial Observations: Dataset Overview

The 20 extracted texts vary substantially in length. The shortest, Andre Norton's *All Cats Are Gray*, contains approximately 2,829 words, while the longest, Mary Shelley's *The Last Man*, contains approximately 174,621.

The dataset also contains multiple works by some authors, particularly Norton, alongside works by other writers. One entry, *Astounding Stories of Super-Science, May 1930*, is a multi-author anthology rather than a single-author novel.

These differences matter for comparing writing styles. Text length may influence the frequency of particular linguistic features, and the uneven representation of authors could affect broader conclusions about authorial style.

Word counts were estimated using Python's `split()` method, which separates text into items wherever spaces or line breaks occur.

### 4.2 Comparing Text Lengths

To investigate variation in text length more clearly, the approximate word counts are visualised in an interactive horizontal bar chart. The works are arranged in ascending order of word count, with book titles as the primary labels and author names shown in the hover information.

This visualisation helps identify unusually short or long works and highlights differences that may need further consideration.

In [16]:

import plotly.io as pio
import plotly.express as px

pio.renderers.default = "notebook_connected"

# Arrange the texts by approximate word count
sorted_books = books_df.sort_values("Words", ascending=True)

# Create an interactive horizontal bar chart
fig = px.bar(
    sorted_books,
    x="Words",
    y="Title",
    orientation="h",
    title="Word Counts Across the 20 Extracted Texts",
    labels={
        "Words": "Approximate Word Count",
        "Title": "Book Title"
    },
    hover_data={
        "Author": True,
        "Filename": True,
        "Words": ":,",
        "Characters": ":,"
    },
    color="Words",
    color_continuous_scale="Blues",
    template="plotly_white"
)

fig.update_layout(
    height=750,
    title_x=0.5,
    title_font_size=18,
    coloraxis_showscale=False,
    margin=dict(l=260, r=40, t=80, b=50)
)

fig.update_xaxes(
    tickformat=",",
    showgrid=True,
    gridcolor="lightgray"
)

fig.update_yaxes(showgrid=False)

fig.show()


#### Observations: Comparing Text Lengths

The bar chart shows considerable variation in the lengths of the 20 extracted texts. Mary Shelley's *The Last Man* is the longest, at approximately 174,621 words, while Andre Norton's *All Cats Are Gray* is the shortest, at approximately 2,829. The longest text is about 62 times the length of the shortest, and *The Last Man* stands out as unusually long within the dataset.

This variation suggests that raw counts of linguistic features may be misleading when comparing writing styles across texts. Normalised measurements, such as frequencies per 1,000 words, will therefore be used in the subsequent analysis.

### 4.3 Exploring Word Frequencies

Word frequency analysis can reveal patterns in the vocabulary used across the dataset. Bird, Klein and Loper (2009) show how Python can be used to examine literary texts by counting word occurrences and identifying frequently used vocabulary, and their discussion of frequency distributions provides a useful starting point for exploring language patterns within the selected works.

The initial analysis examines Mary Shelley's *Frankenstein* as a sample text before the approach is extended to the remaining works. Words are converted to lowercase so that capitalised and uncapitalised versions are counted together, and Python's `Counter` class is then used to identify the most frequently occurring words.

Common function words, such as *the*, *and* and *of*, are retained during this initial exploration because their usage may be relevant to later comparisons of writing styles.

In [17]:

import re
from collections import Counter

# Select a sample text using its Project Gutenberg filename
sample_filename = "84-0.txt"
sample_text = extracted_books[sample_filename]

# Retrieve the book title and author from the metadata
book_title = book_metadata[sample_filename]["Title"]
book_author = book_metadata[sample_filename]["Author"]

# Convert to lowercase and extract words
words = re.findall(r"\b[a-zA-Z]+\b", sample_text.lower())

# Count how often each word appears
word_frequencies = Counter(words)

# Display descriptive information about the analysis
print("WORD FREQUENCY ANALYSIS")
print("-" * 60)
print(f"Book Title: {book_title}")
print(f"Author: {book_author}")
print(f"Total Words Analysed: {len(words):,}")
print(f"Unique Words Identified: {len(word_frequencies):,}")
print("-" * 60)

# Display the 30 most common words
print("\nTop 30 Most Frequently Occurring Words:\n")

for position, (word, count) in enumerate(word_frequencies.most_common(30), start=1):
    print(f"{position:2}. {word:<15} {count:,} occurrences")


WORD FREQUENCY ANALYSIS
------------------------------------------------------------
Book Title: Frankenstein; Or, The Modern Prometheus
Author: Mary Shelley
Total Words Analysed: 75,194
Unique Words Identified: 6,953
------------------------------------------------------------

Top 30 Most Frequently Occurring Words:

 1. the             4,195 occurrences
 2. and             2,976 occurrences
 3. i               2,846 occurrences
 4. of              2,642 occurrences
 5. to              2,089 occurrences
 6. my              1,776 occurrences
 7. a               1,391 occurrences
 8. in              1,128 occurrences
 9. was             1,021 occurrences
10. that            1,018 occurrences
11. me              867 occurrences
12. but             687 occurrences
13. had             686 occurrences
14. with            667 occurrences
15. he              606 occurrences
16. you             572 occurrences
17. which           558 occurrences
18. it              547 occurrences
19. his    

In [18]:

# Create a horizontal bar chart of the 30 most frequent words
top_30_words = word_frequencies.most_common(30)

frequency_df = pd.DataFrame(
    top_30_words,
    columns=["Word", "Frequency"]
)

fig = px.bar(
    frequency_df.sort_values("Frequency"),
    x="Frequency",
    y="Word",
    orientation="h",
    title="Top 30 Most Frequent Words in Frankenstein",
    labels={
        "Frequency": "Number of Occurrences",
        "Word": "Word"
    },
    color="Frequency",
    color_continuous_scale="Blues",
    template="plotly_white"
)

fig.update_layout(
    height=750,
    title_x=0.5,
    coloraxis_showscale=False
)

fig.show()


#### Observations: Word Frequencies in *Frankenstein*

The word-frequency analysis identified 75,194 word occurrences and 6,953 unique words in Mary Shelley's *Frankenstein*.

The bar chart shows that common function words dominate the most frequent vocabulary. The word *the* appears most often (4,195 occurrences), followed by *and* (2,976), which illustrates how a small number of common words account for a substantial proportion of the text. First-person pronouns are also prominent, particularly *I* (2,846), *my* (1,776) and *me* (867). Their frequent use may reflect the first-person narration found throughout the novel, although frequency counts alone cannot explain the structure of the narration.

These findings give an initial indication of vocabulary patterns within one work. Further analysis across the dataset is needed to determine whether similar patterns appear in other texts and whether differences are associated with writing style.

### 4.4 Comparing Word Frequencies Across the Dataset

The initial analysis of *Frankenstein* identified several frequently occurring words, including common function words and first-person pronouns. This section extends the analysis to all 20 works to investigate how their usage varies across the dataset.

Bird, Klein and Loper (2009) show how frequency distributions can be used to examine vocabulary patterns in literary texts. Building on this approach, the ten most frequently occurring words in *Frankenstein* are used as a consistent basis for comparison. Because the texts vary considerably in length, raw frequencies are converted into occurrences per 1,000 words, which allows more meaningful comparisons between works.

In [19]:

# Select the 10 most frequent words from Frankenstein
selected_words = [
    word for word, count in word_frequencies.most_common(10)
]

# Store the results for each book
comparison_results = []

for filename, book_text in extracted_books.items():

    # Extract and count words
    words = re.findall(r"\b[a-zA-Z]+\b", book_text.lower())
    frequencies = Counter(words)
    total_words = len(words)

    # Retrieve book metadata
    result = {
        "Title": book_metadata[filename]["Title"],
        "Author": book_metadata[filename]["Author"]
    }

    # Calculate occurrences per 1,000 words
    for word in selected_words:
        result[word] = (frequencies[word] / total_words) * 1000

    comparison_results.append(result)

# Create the comparison DataFrame
word_comparison_df = pd.DataFrame(comparison_results)

# Display the results
display(word_comparison_df.round(2))


,Title,Author,the,and,i,of,to,my,a,in,was,that
0,Anthem,Ayn Rand,70.94,43.95,5.65,31.44,26.32,4.19,13.55,14.13,4.76,9.00
1,All Cats Are Gray,Andre Norton,51.96,31.17,7.97,27.02,21.82,0.69,23.90,12.82,16.28,11.43
2,"Frankenstein; Or, The Modern Prometheus",Mary Shelley,55.79,39.58,37.85,35.14,27.78,23.62,18.50,15.00,13.58,13.54
3,Plague Ship,Andre Norton,73.04,22.61,3.66,28.89,26.56,0.35,20.58,13.92,15.59,11.77
4,The Last Man,Mary Shelley,64.93,35.50,16.70,40.40,29.25,10.27,14.89,15.91,12.97,10.71
5,Star Born,Andre Norton,74.62,20.84,1.63,30.94,27.04,0.39,20.94,13.56,17.83,14.50
6,Voodoo Planet,Andre Norton,64.90,22.93,6.94,28.34,25.74,0.95,24.00,12.35,13.09,12.76
7,The Gifts of Asti,Andre Norton,82.30,24.74,0.97,41.23,22.47,0.32,21.02,15.20,17.78,8.08
8,Star Hunter,Andre Norton,68.09,17.19,5.18,30.39,24.06,0.57,26.96,13.06,15.75,12.78
9,The Time Traders,Andre Norton,62.48,23.68,3.43,27.63,26.44,0.49,25.40,13.80,16.96,13.19


#### How the Calculation Works

Word-frequency analysis involves counting how often individual words occur within a text. Bird, Klein and Loper (2009) show how frequency distributions can be used to examine vocabulary patterns in literary works, while the Python Software Foundation (n.d.) provides examples of using the `Counter` class and `most_common()` method.

Because the 20 texts vary considerably in length, comparing raw word counts could be misleading, so frequencies are normalised to occurrences per 1,000 words using the following formula:

$$
\text{Frequency per 1,000 words}
=
\frac{\text{Word occurrences}}{\text{Total words}}
\times 1000
$$

For example, the word *I* occurs 2,846 times among the 75,194 words identified in Mary Shelley's *Frankenstein*:

$$
\frac{2846}{75194} \times 1000
\approx 37.85
$$

This means that *I* occurs approximately **37.85 times per 1,000 words** in *Frankenstein*. The same calculation is applied to the ten selected words across all 20 texts, and the normalised frequencies are organised into a pandas DataFrame for comparison.

#### Observations: Comparing Word Frequencies Across the Dataset

Normalised word frequencies show differences in vocabulary use across the 20 texts. Common function words such as *the* and *and* appear throughout, but their rates vary between works.

First-person pronouns show particularly clear differences. The word *I* occurs approximately 56.44 times per 1,000 words in Victoria Glad’s *Each Man Kills*, compared with 37.85 in Mary Shelley’s *Frankenstein* and 1.63 in Andre Norton’s *Star Born*. Differences also appear between works by the same author: Shelley’s *The Last Man* has only 16.70 occurrences per 1,000 words. This suggests that vocabulary patterns can vary between works rather than being determined by authorship alone.

Normalised word frequencies may therefore help identify differences in writing and narrative style, although they cannot explain why those differences occur. Further analysis would be needed to examine how they relate to narrative perspective and storytelling techniques.

# 5. Descriptive Analytics

This section builds on the exploratory analysis of the 20 Project Gutenberg texts by examining writing and storytelling styles in a smaller selection of works.

Six works by three science-fiction authors were selected for closer investigation. Each author has at least two works in the dataset, which allows comparisons both between authors and between works by the same author.

<table>
  <thead>
    <tr>
      <th>Author</th>
      <th>Selected Work 1</th>
      <th>Selected Work 2</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Andre Norton</td>
      <td><em>Plague Ship</em></td>
      <td><em>Star Born</em></td>
    </tr>
    <tr>
      <td>Leigh Brackett</td>
      <td><em>A World Is Born</em></td>
      <td><em>Black Amazon of Mars</em></td>
    </tr>
    <tr>
      <td>Marion Zimmer Bradley</td>
      <td><em>The Door Through Space</em></td>
      <td><em>The Colors of Space</em></td>
    </tr>
  </tbody>
</table>

The analysis examines three aspects of writing style:

- **Sentence length and variation**
- **Use of dialogue**
- **Patterns in common word frequencies**

Although each author is represented by two works, the texts vary considerably in length. Where appropriate, measurements are normalised to allow fairer comparisons, and differences in text length are considered when interpreting the results.

The aim is to identify possible patterns and differences in writing style rather than to establish definitive characteristics of individual authors.


In [20]:
# Select the six works for descriptive analysis
selected_filenames = [
    "pg16921.txt",  # Plague Ship
    "pg18458.txt",  # Star Born
    "pg22544.txt",  # A World Is Born
    "pg32664.txt",  # Black Amazon of Mars
    "pg19726.txt",  # The Door Through Space
    "pg20796.txt"   # The Colors of Space
]

# Display the selected works
selected_books_df = pd.DataFrame([
    {
        "Author": book_metadata[filename]["Author"],
        "Title": book_metadata[filename]["Title"]
    }
    for filename in selected_filenames
])

display(selected_books_df)


,Author,Title
0,Andre Norton,Plague Ship
1,Andre Norton,Star Born
2,Leigh Brackett,A World Is Born
3,Leigh Brackett,Black Amazon of Mars
4,Marion Zimmer Bradley,The Door Through Space
5,Marion Zimmer Bradley,The Colors of Space


## 5.1 Research Question 1: Sentence Style

#### **How do sentence lengths and sentence-length variation differ across the six selected science-fiction works?**

Sentence length is one way of investigating differences in writing style, since some writers may favour longer sentences while others use shorter ones or vary their length more.

This analysis compares the average sentence length and standard deviation for each of the six works. The average shows the typical number of words per sentence, while the standard deviation measures how much sentence lengths vary around that average (Python Software Foundation, n.d.). Sentences are identified using punctuation marks and their words are counted, then the results are compared within and between authors.

This method is only approximate, however, because punctuation does not always mark the end of a sentence, and some introductory publication information may remain after the cleaning process described in Section 3.

In [21]:

import statistics

sentence_results = []

for filename in selected_filenames:
    book_text = extracted_books[filename]

    # Split the text into sentences
    sentences = re.split(r'(?<=[.!?])\s+', book_text)

    # Count the words in each sentence
    sentence_lengths = []

    for sentence in sentences:
        words = re.findall(r"\b[a-zA-Z]+\b", sentence)

        if words:
            sentence_lengths.append(len(words))

    # Calculate sentence-length statistics
    sentence_results.append({
        "Title": book_metadata[filename]["Title"],
        "Author": book_metadata[filename]["Author"],
        "Average Sentence Length": statistics.mean(sentence_lengths),
        "Standard Deviation": statistics.stdev(sentence_lengths)
    })

# Display the results
sentence_df = pd.DataFrame(sentence_results)

display(sentence_df.round(2))


,Title,Author,Average Sentence Length,Standard Deviation
0,Plague Ship,Andre Norton,18.88,10.93
1,Star Born,Andre Norton,18.77,11.33
2,A World Is Born,Leigh Brackett,12.06,7.86
3,Black Amazon of Mars,Leigh Brackett,14.08,9.44
4,The Door Through Space,Marion Zimmer Bradley,16.29,10.70
5,The Colors of Space,Marion Zimmer Bradley,16.33,11.00


In [22]:

# Compare average sentence length across the six works
fig = px.bar(
    sentence_df,
    x="Title",
    y="Average Sentence Length",
    color="Author",
    title="Average Sentence Length Across Selected Science Fiction Works",
    labels={"Average Sentence Length": "Average Words per Sentence"}
)

fig.update_layout(
    xaxis_tickangle=-30
)

fig.show()


#### Observations: Sentence Style

Average sentence length differs across the six works. Andre Norton’s *Plague Ship* and *Star Born* have the longest averages, at 18.88 and 18.77 words respectively, while Leigh Brackett’s works have the shortest, at 12.06 words in *A World Is Born* and 14.08 in *Black Amazon of Mars*.

Works by the same author also show similarities. Marion Zimmer Bradley’s *The Door Through Space* and *The Colors of Space* have almost identical averages (16.29 and 16.33 words), and Norton’s two works are similarly close. Standard deviations range from 7.86 to 11.33 words, indicating that all six works mix shorter and longer sentences rather than keeping a uniform length.

Sentence length may therefore help distinguish the selected authors’ styles. However, the analysis covers only two works per author and relies on approximate sentence splitting, so the results should not be treated as definitive characteristics of each author’s writing.

## 5.2 Research Question 2: Dialogue and Storytelling Techniques

#### **How does the estimated proportion of dialogue differ across the six selected science-fiction works?**

Dialogue is an important storytelling technique because it allows characters to speak directly and can affect how a story is presented.

This analysis estimates how much of each selected work consists of dialogue. Python's regular expression functions are used to identify passages enclosed in double quotation marks (Python Software Foundation, n.d.-d), with matches restricted to single paragraphs to avoid capturing long stretches of narration. Words within quoted passages are counted and expressed as a percentage of each work’s total, allowing comparison across texts of different lengths.

However, quotation marks do not always represent spoken dialogue, and some dialogue may be missed because of differences in formatting. The results are therefore treated as estimates rather than exact measurements.

In [23]:

dialogue_results = []

for filename in selected_filenames:
    book_text = extracted_books[filename]

    # Find quoted passages without crossing paragraph breaks
    dialogue_passages = re.findall(
        r'"((?:(?!\n\s*\n)[^"])*?)"',
        book_text
    )

    # Count words in detected dialogue
    dialogue_words = sum(
        len(re.findall(r"\b[a-zA-Z]+\b", passage))
        for passage in dialogue_passages
    )

    # Count all words
    total_words = len(re.findall(r"\b[a-zA-Z]+\b", book_text))

    dialogue_results.append({
        "Title": book_metadata[filename]["Title"],
        "Author": book_metadata[filename]["Author"],
        "Estimated Dialogue (%)": (dialogue_words / total_words) * 100
    })

dialogue_df = pd.DataFrame(dialogue_results)

display(dialogue_df.round(2))


,Title,Author,Estimated Dialogue (%)
0,Plague Ship,Andre Norton,19.56
1,Star Born,Andre Norton,12.30
2,A World Is Born,Leigh Brackett,23.20
3,Black Amazon of Mars,Leigh Brackett,16.43
4,The Door Through Space,Marion Zimmer Bradley,21.63
5,The Colors of Space,Marion Zimmer Bradley,36.17


In [24]:

# Compare estimated dialogue percentages across the six works
fig = px.bar(
    dialogue_df,
    x="Title",
    y="Estimated Dialogue (%)",
    color="Author",
    title="Estimated Dialogue Proportions Across Selected Science Fiction Works",
    labels={"Estimated Dialogue (%)": "Estimated Dialogue (%)"}
)

fig.update_layout(
    xaxis_tickangle=-30
)

fig.show()


#### Observations: Dialogue and Storytelling Techniques

The estimated proportion of dialogue differs across the six works. Marion Zimmer Bradley’s *The Colors of Space* has the highest estimate at 36.17%, while Andre Norton’s *Star Born* has the lowest at 12.30%.

Estimates also differ between works by the same author. Bradley’s *The Door Through Space* is at 21.63% compared with 36.17% for *The Colors of Space*, Norton’s *Plague Ship* and *Star Born* are at 19.56% and 12.30%, and Leigh Brackett’s *A World Is Born* and *Black Amazon of Mars* are at 23.20% and 16.43%.

The use of dialogue may therefore vary both between authors and between individual works. However, because dialogue is identified as quoted text within a single paragraph, the method may miss dialogue that spans several paragraphs or include quotations that are not spoken dialogue. The results should be interpreted as approximate rather than exact measurements.

## 5.3 Research Question 3: Authorial Fingerprints

#### **Are patterns in common word usage more similar between works by the same author than between works by different authors?**

An authorial fingerprint refers to recurring language patterns that may help distinguish one writer’s style from another, and research in authorship analysis has shown that frequently used words can be useful for comparing styles (Burrows, 2002).

Building on the normalised word-frequency analysis in Section 4.4, this section compares six common function words: *the*, *and*, *of*, *to*, *in* and *that*. Frequencies are calculated per 1,000 words so that texts of different lengths can be compared, and the results are displayed in a heatmap to show similarities and differences in word usage across the six works.

However, word frequencies may also be affected by narrative perspective, subject matter and individual storytelling choices. With only two works per author, the findings are treated as possible stylistic patterns rather than definitive evidence of authorship.

In [25]:

# Select common function words for authorial fingerprint analysis
function_words = ["the", "and", "of", "to", "in", "that"]

print("Selected function words:", ", ".join(function_words))


Selected function words: the, and, of, to, in, that


In [26]:

# Calculate normalised function-word frequencies
fingerprint_results = []

for filename in selected_filenames:
    book_text = extracted_books[filename]

    # Convert text to lowercase and identify words
    words = re.findall(r"\b[a-zA-Z]+\b", book_text.lower())
    frequencies = Counter(words)
    total_words = len(words)

    # Store the book details
    result = {
        "Title": book_metadata[filename]["Title"],
        "Author": book_metadata[filename]["Author"]
    }

    # Calculate occurrences per 1,000 words
    for word in function_words:
        result[word] = (frequencies[word] / total_words) * 1000

    fingerprint_results.append(result)

# Display the comparison table
fingerprint_df = pd.DataFrame(fingerprint_results)

display(fingerprint_df.round(2))


,Title,Author,the,and,of,to,in,that
0,Plague Ship,Andre Norton,73.04,22.61,28.89,26.56,13.92,11.77
1,Star Born,Andre Norton,74.62,20.84,30.94,27.04,13.56,14.50
2,A World Is Born,Leigh Brackett,69.79,22.83,29.93,18.33,13.13,8.52
3,Black Amazon of Mars,Leigh Brackett,77.06,39.19,35.10,15.17,13.19,14.06
4,The Door Through Space,Marion Zimmer Bradley,57.03,33.51,24.82,18.05,15.85,8.65
5,The Colors of Space,Marion Zimmer Bradley,51.64,25.74,21.50,21.45,13.19,9.98


In [27]:

# Prepare the function-word frequencies for a heatmap
heatmap_data = fingerprint_df.set_index("Title")[function_words]

# Create a heatmap
fig = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    labels={
        "x": "Function Word",
        "y": "Book",
        "color": "Occurrences per 1,000 Words"
    },
    title="Function-Word Frequencies Across Selected Science Fiction Works"
)

fig.show()


#### Measuring Similarity Between Works

To investigate whether works by the same author have more similar function-word patterns, pairwise distance (Euclidean) was calculated between every possible pair of books. Each book was represented by its normalised frequencies for the six selected function words, so a smaller distance indicates greater similarity and a larger distance indicates greater difference. The calculation used NumPy’s `linalg.norm` function (NumPy Developers, n.d.).

The 15 possible pairs comprised three comparisons between works by the same author and twelve between works by different authors. The mean, minimum and maximum distances were calculated for each group.

This is a simple exploratory comparison rather than a formal authorship identification test. The distances depend on the six words selected and may be influenced more strongly by words with larger frequency differences.

In [28]:

import numpy as np
from itertools import combinations

# Compare every possible pair of selected books
distance_results = []

for i, j in combinations(range(len(fingerprint_df)), 2):

    # Get the six function-word frequencies for each book
    book1 = fingerprint_df.loc[i, function_words].to_numpy(dtype=float)
    book2 = fingerprint_df.loc[j, function_words].to_numpy(dtype=float)

    # Calculate Euclidean distance
    distance = np.linalg.norm(book1 - book2)

    # Check whether the books share an author
    same_author = (
        fingerprint_df.loc[i, "Author"] ==
        fingerprint_df.loc[j, "Author"]
    )

    distance_results.append({
        "Book 1": fingerprint_df.loc[i, "Title"],
        "Book 2": fingerprint_df.loc[j, "Title"],
        "Comparison": "Same author" if same_author else "Different authors",
        "Distance": distance
    })

distance_df = pd.DataFrame(distance_results)

# Summarise the distances by comparison type
distance_summary = (
    distance_df.groupby("Comparison")["Distance"]
    .agg(["mean", "min", "max", "count"])
    .round(2)
)

display(distance_summary)


,mean,min,max,count
Comparison,,,,
Different authors,21.42,9.52,32.68,12
Same author,11.63,4.19,19.70,3


#### Observations: Authorial Fingerprints

The heatmap shows that Andre Norton’s *Plague Ship* and *Star Born* have closely matching frequencies for the six function words, while Leigh Brackett’s works show greater variation.

The pairwise distance (Euclidean) results support this pattern. The mean distance between works by the same author was 11.63, compared with 21.42 between works by different authors, suggesting that works by the same author generally have more similar function-word patterns within this selection. However, the ranges overlap: the minimum distance between works by different authors was 9.52, while the maximum between works by the same author was 19.70, so some works by different authors are more alike than certain works by the same author.

Function-word frequencies may therefore provide useful indicators of authorial style. However, the analysis covers only six works, two per author, and six function words, so the results should be interpreted as exploratory evidence of stylistic patterns rather than a reliable method of identifying authorship.

# 6. Suggestions

The analysis identified differences in sentence length, estimated dialogue proportions and common word usage across the six selected works. However, several improvements could make future investigations more reliable and give a clearer picture of individual writing styles.

### 6.1 Increase the Number of Texts

A future study could include more works by each author. With only two works per author, it is difficult to tell whether the patterns found are typical of their writing. A larger selection would allow more comparisons and show whether the observed differences are consistent.

### 6.2 Improve Sentence and Dialogue Detection

Sentence boundaries were identified using punctuation and dialogue using quotation marks. Both methods are approximate and may produce errors where texts contain unusual punctuation or formatting. Natural language processing tools, such as those described by Bird, Klein and Loper (2009), could improve sentence detection, and examining how each text formats spoken passages could improve dialogue detection.

### 6.3 Expand the Authorial Fingerprint Analysis

The authorial fingerprint analysis compared only six function words. Although works by the same author were more similar on average, this was based on a limited selection. Future work could use a larger set of frequently used words and compare different measures of stylistic similarity, giving a more detailed picture of the recurring language patterns associated with each author (Burrows, 2002).

### 6.4 Improve Text Cleaning and Consistency

Some publication information may remain in the texts after cleaning, which could affect word counts and other measurements. More consistent methods for removing introductory material, along with checks on the extracted text before calculations, would help ensure the results reflect the literary works themselves.

### 6.5 Overall Recommendation

The most useful next step would be to expand the dataset while improving the methods used to identify sentences and dialogue, which would show whether the patterns in this small selection hold across a wider range of works.

The findings show how computational methods can support the study of literary style, but they should be used with an awareness of the limitations of automated text analysis.


## 7. Bibliography

Bird, S., Klein, E. and Loper, E. (2009) *Natural Language Processing with Python*. 1st edn. Sebastopol, CA: O'Reilly Media.

Burrows, J. (2002) '"Delta": a measure of stylistic difference and a guide to likely authorship', *Literary and Linguistic Computing*, 17(3), pp. 267–287. Available at: https://doi.org/10.1093/llc/17.3.267

Johnson, G.B. (n.d.) *Gutendex: Web API for Project Gutenberg ebook metadata*. Available at: https://github.com/garethbjohnson/gutendex (Accessed: 11 November 2026).

NumPy Developers (n.d.) *numpy.linalg.norm*. Available at: https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html (Accessed: 11 November 2026).

Plotly Technologies Inc. (n.d.-a) *Bar charts in Python*. Available at: https://plotly.com/python/bar-charts/ (Accessed: 11 November 2026).

Plotly Technologies Inc. (n.d.-b) *Displaying figures in Python*. Available at: https://plotly.com/python/renderers/ (Accessed: 11 November 2026).

Project Gutenberg (n.d.) *Help on the bibliographic record*. Available at: https://www.gutenberg.org/help/bibliographic_record.html (Accessed: 11 November 2026).

Project Gutenberg (n.d.) *Offline catalogs*. Available at: https://www.gutenberg.org/ebooks/offline_catalogs.html (Accessed: 11 November 2026).

Python Software Foundation (n.d.) *5. Data structures*. Available at: https://docs.python.org/3/tutorial/datastructures.html (Accessed: 11 November 2026).

Python Software Foundation (n.d.) *Built-in Types — str.find()*. Python Documentation. Available at: https://docs.python.org/3/library/stdtypes.html#str.find (Accessed: 10 November 2026).

Python Software Foundation (n.d.) *collections — Container datatypes*. Available at: https://docs.python.org/3/library/collections.html#collections.Counter (Accessed: 10 November 2026).

Python Software Foundation (n.d.) *Built-in Types — String Methods*. Python Documentation. Available at: https://docs.python.org/3/library/stdtypes.html#string-methods (Accessed: 10 November 2026).

Python Software Foundation (n.d.) *re — Regular expression operations*. Available at: https://docs.python.org/3/library/re.html (Accessed: 11 November 2026).

Python Software Foundation (n.d.) *statistics — Mathematical statistics functions*. Available at: https://docs.python.org/3/library/statistics.html (Accessed: 11 November 2026).
